# 📊 实验四：优化前后对比与结论

## 学习目标

1. 加载实验二、实验三保存的结果
2. 对比时延、吞吐、显存与算子数量
3. 总结算子融合的适用场景与收益

In [ ]:
import json
import os

results_dir = "results"
required = ["baseline.json", "optimized.json"]
missing = [f for f in required if not os.path.exists(os.path.join(results_dir, f))]
if missing:
    raise FileNotFoundError(f"缺少结果文件 {missing}，请先运行 02 和 03 两个 notebook")

with open(os.path.join(results_dir, "baseline.json"), encoding="utf-8") as f:
    baseline = json.load(f)
with open(os.path.join(results_dir, "optimized.json"), encoding="utf-8") as f:
    optimized = json.load(f)

print(f"设备: {baseline.get('device')} / {optimized.get('device')}")

In [ ]:
metrics = [
    ("ms_per_iter", "平均时延 (ms/iter)"),
    ("throughput", "吞吐 (images/s)"),
    ("peak_memory_mb", "峰值显存 (MB)"),
]

print(f"{'指标':<22}{'优化前':>16}{'优化后':>16}{'变化':>14}")
for key, label in metrics:
    before = baseline.get(key)
    after = optimized.get(key)
    if before is None or after is None:
        continue
    change = (after - before) / before * 100
    print(f"{label:<22}{before:>16.2f}{after:>16.2f}{change:>+13.1f}%")

before_ops = baseline["module_counts"]
after_ops = optimized["module_counts"]
print(f"算子总数（Conv+BN）: {before_ops['total']} -> {after_ops['total']}（减少 {before_ops['total'] - after_ops['total']}）")
print(f"其中 BN: {before_ops['batchnorm2d']} -> {after_ops['batchnorm2d']}")

In [ ]:
import matplotlib.pyplot as plt

labels = ["Before", "After"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

latency = [baseline["ms_per_iter"], optimized["ms_per_iter"]]
throughput = [baseline["throughput"], optimized["throughput"]]
memory = [baseline.get("peak_memory_mb") or 0, optimized.get("peak_memory_mb") or 0]
ops = [before_ops["total"], after_ops["total"]]

for ax, title, values, color in zip(
    axes,
    ["Latency (ms/iter)", "Throughput (images/s)", "Peak Memory (MB)", "Operator Count"],
    [latency, throughput, memory, ops],
    ["#3498db", "#2ecc71", "#e67e22", "#e74c3c"],
):
    bars = ax.bar(labels, values, color=color)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, v, f"{v:.2f}", ha="center", va="bottom")
    ax.set_title(title)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 实验结论

1. Conv+BN 融合是**数学等价**的算子优化，融合后输出与原始模型一致（浮点舍入误差除外）
2. 融合后 BN 算子被全部移除，算子总数下降，kernel 启动和中间张量读写减少
3. 实际时延 / 吞吐提升幅度取决于设备与 batch size；NPU 上建议再结合 CANN 图优化与混合精度
4. 该优化只适用于推理（eval）阶段；训练阶段 BN 依赖 batch 统计量，不能直接折叠
5. 本次实验完成了"定位算子 → 分析原理 → 实施优化 → 前后对比"的完整闭环

## 实验报告要点

- 实验设备与软件版本
- 选择的算子与优化方法
- 优化前后算子数量、时延、吞吐、显存数据
- 融合前后正确性校验结果
- 结论与可推广的优化思路

## 课后练习

1. (单选题) 优化后吞吐提升 15%、测量噪声 ±5%，正确结论是？
   - A. 有提升趋势，但需更多 repeats/置信度验证
   - B. 必然提升
   - C. 完全无效
   - D. 只能说明显存降低

2. (单选题) 优化后时延下降但峰值显存不变，最可能原因？
   - A. 减少了 kernel 启动与中间读写，但中间张量峰值未下降
   - B. 参数减少
   - C. 精度下降
   - D. 数据增强

3. (多选题) 对比报告必须说明？
   - A. 硬件型号与软件版本
   - B. batch_size 与输入
   - C. warmup/repeats 与同步
   - D. 结论适用范围

4. (多选题) 融合后还可叠加的优化？
   - A. FP16/BF16
   - B. CANN 图优化
   - C. 更大 batch
   - D. 增加 BN 层

5. (判断题) baseline 与 optimized 使用不同 batch_size 也可以直接比较吞吐。

6. (判断题) 模型层融合与 CANN/ATC 图优化可以叠加。

7. (填空题) 相对提升率公式：speedup = ____。

8. (填空题) 报告应记录设备型号、____、____ 等环境信息。

9. (简答题) 融合后没有明显收益的可能原因有哪些？

10. (简答题) 如何量化融合前后数值误差？

11. (代码设计题) 编写 compare_results(baseline.json, optimized.json)，输出各指标相对变化百分比。

12. (单选题) 优化后时延下降且精度未变，可以认为？
   - A. 该优化在本实验设定下有效
   - B. 优化无效
   - C. 必须重训
   - D. 数据错误

13. (多选题) 对比实验中需要保持的变量包括？
   - A. 输入分布
   - B. batch_size
   - C. warmup/repeats
   - D. 设备状态

14. (判断题) 单次 batch 的测量结果足以作为最终性能结论。

15. (简答题) 总结一套可复用于其他算子的优化方法论。

> 参考答案见 answer/06.06_comparison_and_conclusion_answer.ipynb。